# Phase 2: LSTM system-identification dataset

Generate reproducible, mostly healthy trajectories from the validated nonlinear DC motor plant. Runs are split before overlapping windows are created to prevent leakage.

In [1]:
from dataclasses import asdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from data_utils import (
    build_training_sequences,
    fit_normalization,
    generate_trajectory,
    make_multisine_voltage,
    make_random_step_signal,
    normalize_sequences,
    save_processed_dataset,
    split_runs,
    vary_motor_params,
)
from motor_model import DCMotorParams

## Reproducible generation settings

In [2]:
SEED = 2026
NUMBER_OF_RUNS = 30
DURATION = 12.0
TIMESTEP = 0.01
WINDOW_LENGTH = 20
HORIZON = 1
SPEED_NOISE_STD = 0.25  # rad/s
VOLTAGE_LIMITS = (0.0, 12.0)
LOAD_LEVELS = (0.0, 0.03, 0.06, 0.09)
PARAMETER_VARIATION = 0.10

rng = np.random.default_rng(SEED)
split_rng = np.random.default_rng(SEED + 1)
nominal_params = DCMotorParams()

## Generate trajectories

Runs alternate between multilevel random-step voltage and bounded multi-sine voltage. Every fifth run uses exactly nominal motor parameters; the others use independent ±10% parameter variation. Load torque changes between realistic levels every 3 seconds.

In [3]:
trajectories = []
run_parameters = []
excitation_types = []
step_durations = (0.2, 0.35, 0.5)
voltage_levels = np.linspace(*VOLTAGE_LIMITS, 5)

for run_id in range(NUMBER_OF_RUNS):
    params = (
        nominal_params
        if run_id % 5 == 0
        else vary_motor_params(nominal_params, rng, PARAMETER_VARIATION)
    )
    if run_id % 2 == 0:
        voltage = make_random_step_signal(
            rng, DURATION, step_durations[(run_id // 2) % 3], voltage_levels
        )
        excitation_type = 'random_step'
    else:
        voltage = make_multisine_voltage(rng, VOLTAGE_LIMITS)
        excitation_type = 'multisine'

    load_torque = (
        (lambda _time: 0.0)
        if run_id == 0
        else make_random_step_signal(rng, DURATION, 3.0, LOAD_LEVELS)
    )
    trajectories.append(
        generate_trajectory(
            run_id,
            voltage,
            load_torque,
            params,
            rng,
            duration=DURATION,
            timestep=TIMESTEP,
            speed_noise_std=SPEED_NOISE_STD,
        )
    )
    run_parameters.append(params)
    excitation_types.append(excitation_type)

print(f'Generated {len(trajectories)} runs with {len(trajectories[0]["time"])} samples each.')

Generated 30 runs with 1201 samples each.


## Representative excitation and response

In [4]:
fig, axes = plt.subplots(3, 2, sharex='col', figsize=(12, 7))
for column, run_index in enumerate((0, 1)):
    run = trajectories[run_index]
    axes[0, column].plot(run['time'], run['voltage'])
    axes[0, column].set_title(excitation_types[run_index].replace('_', ' ').title())
    axes[0, column].set_ylabel('Voltage (V)')
    axes[1, column].plot(run['time'], run['y_true'], label='true')
    axes[1, column].plot(run['time'], run['y_measured'], alpha=0.45, label='measured')
    axes[1, column].set_ylabel('Speed (rad/s)')
    axes[1, column].legend()
    axes[2, column].plot(run['time'], run['load_torque'])
    axes[2, column].set_ylabel('Load (N m)')
    axes[2, column].set_xlabel('Time (s)')
for axis in axes.flat:
    axis.grid(alpha=0.3)
fig.tight_layout()
plt.show()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_7564\2490666775.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Split runs, then create windows

The 60/20/20 split is applied to complete trajectories. Only then are overlapping windows constructed inside each split.

In [5]:
splits = split_runs(trajectories, split_rng)
windowed = {}

for split_name, split_trajectories in splits.items():
    run_windows = [
        build_training_sequences(run, WINDOW_LENGTH, HORIZON)
        for run in split_trajectories
    ]
    windowed[split_name] = {
        'inputs': np.concatenate([item[0] for item in run_windows]),
        'targets': np.concatenate([item[1] for item in run_windows]),
        'run_ids': np.concatenate([item[2] for item in run_windows]),
    }

## Fit training-only normalization

In [6]:
normalization = fit_normalization(
    windowed['train']['inputs'], windowed['train']['targets']
)
normalized = {}
for split_name, arrays in windowed.items():
    inputs, targets = normalize_sequences(
        arrays['inputs'], arrays['targets'], normalization
    )
    normalized[split_name] = {'inputs': inputs, 'targets': targets}

## Input and output distributions

In [7]:
train_samples = {
    key: np.concatenate([run[key] for run in splits['train']])
    for key in ('voltage', 'y_measured', 'y_true')
}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for axis, key, label in zip(
    axes,
    ('voltage', 'y_measured', 'y_true'),
    ('Voltage input (V)', 'Measured speed input (rad/s)', 'True speed target (rad/s)'),
):
    axis.hist(train_samples[key], bins=50, alpha=0.8)
    axis.set_xlabel(label)
    axis.set_ylabel('Count')
    axis.grid(alpha=0.2)
fig.tight_layout()
plt.show()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_7564\598443716.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Example LSTM windows

In [8]:
example_indices = (0, len(windowed['train']['inputs']) // 2, -1)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, index in zip(axes, example_indices):
    window = windowed['train']['inputs'][index]
    axis.plot(window[:, 0], color='tab:blue')
    axis.set_xlabel('Window step')
    axis.set_ylabel('Voltage (V)', color='tab:blue')
    speed_axis = axis.twinx()
    speed_axis.plot(window[:, 1], color='tab:orange')
    speed_axis.set_ylabel('Measured speed (rad/s)', color='tab:orange')
    axis.set_title(f'Run {windowed["train"]["run_ids"][index]}')
    axis.grid(alpha=0.3)
fig.tight_layout()
plt.show()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_7564\3190963376.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Dataset sizes and sanity checks

In [9]:
for split_name in ('train', 'validation', 'test'):
    print(
        f'{split_name:10s}: {len(splits[split_name]):2d} runs, '
        f'X {normalized[split_name]["inputs"].shape}, '
        f'y {normalized[split_name]["targets"].shape}'
    )

numeric_fields = ('time', 'voltage', 'current', 'y_true', 'y_measured', 'rpm', 'load_torque')
assert all(
    np.isfinite(run[field]).all()
    for run in trajectories
    for field in numeric_fields
)
assert all(
    run['voltage'].min() >= VOLTAGE_LIMITS[0] - 1e-9
    and run['voltage'].max() <= VOLTAGE_LIMITS[1] + 1e-9
    for run in trajectories
)

split_ids = {
    name: {int(run['run_id'][0]) for run in runs}
    for name, runs in splits.items()
}
assert split_ids['train'].isdisjoint(split_ids['validation'])
assert split_ids['train'].isdisjoint(split_ids['test'])
assert split_ids['validation'].isdisjoint(split_ids['test'])
assert len(set.union(*split_ids.values())) == NUMBER_OF_RUNS
assert set(windowed['train']['run_ids']).isdisjoint(windowed['test']['run_ids'])

train_inputs = normalized['train']['inputs']
train_targets = normalized['train']['targets']
assert np.allclose(train_inputs.mean(axis=(0, 1), dtype=np.float64), 0.0, atol=1e-4)
assert np.allclose(train_inputs.std(axis=(0, 1), dtype=np.float64), 1.0, atol=1e-4)
assert np.allclose(train_targets.mean(axis=0, dtype=np.float64), 0.0, atol=1e-4)
assert np.allclose(train_targets.std(axis=0, dtype=np.float64), 1.0, atol=1e-4)

for split_name in splits:
    inputs = normalized[split_name]['inputs']
    targets = normalized[split_name]['targets']
    assert inputs.ndim == 3 and inputs.shape[1:] == (WINDOW_LENGTH, 2)
    assert targets.shape == (len(inputs), HORIZON)
    assert np.isfinite(inputs).all() and np.isfinite(targets).all()

print('All dataset sanity checks passed.')

train     : 18 runs, X (21258, 20, 2), y (21258, 1)
validation:  6 runs, X (7086, 20, 2), y (7086, 1)
test      :  6 runs, X (7086, 20, 2), y (7086, 1)
All dataset sanity checks passed.


## Save the processed dataset

In [10]:
parameter_names = np.array(list(asdict(nominal_params)))
parameter_values = np.array(
    [[getattr(params, name) for name in parameter_names] for params in run_parameters],
    dtype=np.float32,
)
dataset = {
    'time': trajectories[0]['time'].astype(np.float32),
    'run_ids': np.arange(NUMBER_OF_RUNS, dtype=np.int32),
    'excitation_type': np.array(excitation_types),
    'parameter_names': parameter_names,
    'parameter_values': parameter_values,
    'feature_names': np.array(['voltage', 'y_measured']),
    'target_name': np.array('y_true'),
    'window_length': np.array(WINDOW_LENGTH),
    'horizon': np.array(HORIZON),
    'timestep': np.array(TIMESTEP),
    'duration': np.array(DURATION),
    'seed': np.array(SEED),
    'speed_noise_std': np.array(SPEED_NOISE_STD),
    'voltage_limits': np.array(VOLTAGE_LIMITS),
}
for field in ('voltage', 'current', 'y_true', 'y_measured', 'rpm', 'load_torque'):
    dataset[field] = np.stack([run[field] for run in trajectories]).astype(np.float32)
for name, values in normalization.items():
    dataset[f'normalization_{name}'] = values
for split_name in ('train', 'validation', 'test'):
    dataset[f'X_{split_name}'] = normalized[split_name]['inputs']
    dataset[f'y_{split_name}'] = normalized[split_name]['targets']
    dataset[f'window_run_ids_{split_name}'] = windowed[split_name]['run_ids']
    dataset[f'split_run_ids_{split_name}'] = np.array(
        sorted(split_ids[split_name]), dtype=np.int32
    )

dataset_path = project_root / 'data' / 'processed' / 'dc_motor_lstm_dataset.npz'
save_processed_dataset(dataset_path, **dataset)
with np.load(dataset_path) as saved:
    assert saved['X_train'].shape == normalized['train']['inputs'].shape
    assert saved['normalization_input_mean'].shape == (2,)
print(f'Saved {dataset_path} ({dataset_path.stat().st_size / 1e6:.2f} MB)')

Saved C:\Users\Lakshya\OneDrive\Desktop\cs\project\data\processed\dc_motor_lstm_dataset.npz (0.99 MB)


## Phase 2 summary

The dataset contains **30 reproducible 12-second runs** sampled every **0.01 s**. Half use multilevel random-step (PRBS-style) voltage with several hold times and half use bounded multi-sine voltage; every signal remains within **0–12 V**. Runs cover load torque levels from **0 to 0.09 N m**, exact nominal parameters every fifth run, and independent **±10%** parameter variation otherwise. Only small Gaussian speed measurement noise (standard deviation **0.25 rad/s**) is added; no severe sensor faults are included.

Complete runs are split **60/20/20** into **18 training, 6 validation, and 6 test trajectories** before windowing. Each length-**20** input window contains `[voltage, measured speed]`; its one-step-ahead target is true motor speed, stored with shape `(samples, 1)`. The horizon argument can later produce multi-step targets without changing the split logic. Feature-wise input statistics and target statistics are fitted only on training windows, applied to every split, and saved with the normalized arrays in `data/processed/dc_motor_lstm_dataset.npz`. The archive also retains time, voltage, current, true and measured speed, RPM, load torque, run IDs, excitation types, and parameter values.